# Building a Reflexion Agent with External Knowledge Integration

This project builds a research agent using the **Reflexion** pattern. Rather than answering a question once, the agent drafts an answer, critiques its own draft to find what is missing or superfluous, proposes search queries, retrieves real evidence from the web, and then revises the answer with citations — looping until it hits an iteration limit.

The result is an agent that grounds its output in sources it actually retrieved instead of relying purely on what the model already knows. The worked example researches nutrition questions, but the same graph works for any research task: swap the prompts and the question.

> **Note:** this is a demonstration of an agent architecture, not a source of professional advice. Model output can be wrong or incomplete — verify anything important against the cited sources.

## Table of Contents

1. What this project covers
2. Setup and API keys
3. The web-search tool
4. The LLM and the prompt
5. The responder and its structured output
6. Tool execution
7. The revisor
8. Building the graph
9. Running the agent

## What this project covers

- The core principles of the **Reflexion** framework
- Building an agent that critiques and improves its own responses
- Using LangGraph to create a cyclical, iterative agent workflow
- Integrating an external web-search tool into a LangChain agent
- Forcing structured agent output with Pydantic models and tool binding
- Constructing layered prompts for nuanced agent behavior

----


## Setup


This project uses the following libraries:

* [`langchain`](https://www.langchain.com/) — core LangChain functionality.
* [`langchain-groq`](https://pypi.org/project/langchain-groq/) — Groq-hosted LLMs.
* [`langchain-community`](https://pypi.org/project/langchain-community/) — the Serper search wrapper.
* [`langgraph`](https://langchain-ai.github.io/langgraph/) — the cyclical Reflexion workflow.
* [`pydantic`](https://docs.pydantic.dev/) — schemas that force structured agent output.
* [`python-dotenv`](https://pypi.org/project/python-dotenv/) — load API keys from `.env`.

### Installing Required Libraries
Run the following to install the required libraries (it might take a few minutes):


In [ ]:
%%capture
%pip install langchain langchain-groq langchain-community langgraph pydantic python-dotenv
%pip install langchain-tavily   # optional: only needed if you use Tavily for search

### Importing Required Libraries



In [ ]:
import os
import json
from typing import List, Dict

from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage
from langchain_groq import ChatGroq
from langgraph.graph import END, MessageGraph

---


## Writing the Code


### API keys

This agent needs an LLM key and **one** search key — all have free tiers:

* `GROQ_API_KEY` — the LLM ([console.groq.com/keys](https://console.groq.com/keys))
* `TAVILY_API_KEY` — search, preferred ([tavily.com](https://app.tavily.com/sign-in))
* `SERPER_API_KEY` — search, used if no Tavily key is set ([serper.dev](https://serper.dev))

Put them in a `.env` file in this folder (or the repo root); the next cell loads them and checks that what you need is present.

In [ ]:
# Load API keys from a .env file (this folder or any parent directory).
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("GROQ_API_KEY"):
    raise RuntimeError("GROQ_API_KEY is not set. Add it to your .env file before running.")

# Either search provider works; Tavily is preferred when available.
if not (os.environ.get("TAVILY_API_KEY") or os.environ.get("SERPER_API_KEY")):
    raise RuntimeError("Set TAVILY_API_KEY or SERPER_API_KEY in your .env file.")

print("API keys loaded.")

### Tool setup: web search

The agent needs a way to find information. The helper below wraps whichever provider you have configured — **Tavily** when `TAVILY_API_KEY` is set (it is built for agent retrieval and returns cleaner page content), otherwise **Serper**, which queries Google.

Both are normalized to the same `{title, link, snippet}` shape, so the rest of the notebook does not care which one is running. Snippets are trimmed to keep the payload handed back to the model small.

In [ ]:
# Web-search tool. Tavily is built for agent retrieval and is used when its key is
# present; otherwise we fall back to Serper. Both are normalized to the same shape,
# so everything downstream is provider-agnostic.
if os.environ.get("TAVILY_API_KEY"):
    from langchain_tavily import TavilySearch

    _tavily = TavilySearch(max_results=3)
    SEARCH_PROVIDER = "tavily"

    def web_search(query: str) -> List[Dict[str, str]]:
        """Run a web search and return a list of {title, link, snippet}."""
        hits = _tavily.invoke({"query": query})
        if isinstance(hits, dict):          # newer versions wrap results in a dict
            hits = hits.get("results", [])
        return [
            {
                "title": h.get("title", ""),
                "link": h.get("url", ""),
                "snippet": (h.get("content", "") or "")[:300],
            }
            for h in hits[:3]
        ]

else:
    from langchain_community.utilities import GoogleSerperAPIWrapper

    _serper = GoogleSerperAPIWrapper(k=3)
    SEARCH_PROVIDER = "serper"

    def web_search(query: str) -> List[Dict[str, str]]:
        """Run a web search and return a list of {title, link, snippet}."""
        hits = _serper.results(query).get("organic", [])
        return [
            {
                "title": h.get("title", ""),
                "link": h.get("link", ""),
                "snippet": h.get("snippet", ""),
            }
            for h in hits[:3]
        ]

print(f"search provider: {SEARCH_PROVIDER}")

# Quick sanity check of the tool on its own.
for hit in web_search("healthy breakfast options"):
    print(" -", hit["title"], "->", hit["link"])

### LLM and Prompting

At the core of the agent is a large language model — here a Groq-hosted Llama model. First, let's see how the standalone LLM answers a question with no special prompting or tools, so we have a baseline to compare the Reflexion loop against:

In [ ]:
# The model behind both the responder and the revisor.
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

question = "What does current evidence say about breakfast choices for blood-sugar management?"
print(llm.invoke(question).content[:500])

#### Crafting the Agent's Persona and Logic

To steer the agent's behavior we build a detailed prompt template. It gives the model a clear role — an evidence-based nutrition researcher — plus a fixed set of steps to follow, so its responses stay consistent and always carry the reflection logic.

The prompt instructs the agent to:
1. Provide an initial answer.
2. Explain the physiological rationale behind it.
3. Note where the evidence is contested, representing opposing views fairly.
4. **Reflect on and critique** its own answer.
5. Generate **search queries** to fill the gaps it identified.

In [ ]:
prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an evidence-based nutrition researcher. You summarize what the published
        literature actually supports, represent competing viewpoints fairly, and are explicit
        about uncertainty and the limits of the evidence.

        Your response must follow these steps:
        1. {first_instruction}
        2. Explain the physiological rationale behind your answer, naming the mechanisms involved.
        3. Note where the evidence is contested or weak, and represent the main opposing views fairly.
        4. Reflect on and critique your own answer: what is missing, and what is unnecessary?
        5. After the reflection, **list 1-3 search queries separately** that would help verify or
           extend the answer. Do not include them inside the reflection.

        Prefer systematic reviews and randomized trials over individual studies or anecdote.
        """
    ),
    MessagesPlaceholder(variable_name="messages"),
    (
        "system",
        "Answer the user's question above using the required format."
    ),
])

### Defining the Responder

The **Responder** is the first component of our agent's thinking process. It generates the initial draft of the answer based on the user's question and the persona we defined in the prompt.

Here, we create a chain that combines our prompt template with the LLM. We then invoke it with our sample question to see the initial, un-critiqued response:


In [ ]:
first_responder_prompt = prompt_template.partial(first_instruction="Provide a detailed ~250 word answer")
temp_chain = first_responder_prompt| llm
response = temp_chain.invoke({"messages": [HumanMessage(content=question)]})
print(response.content)

#### Structuring the Agent's Output: Data Models

To make the agent's self-critique process reliable, we need to enforce a specific output structure. We use Pydantic `BaseModel` to define two data classes:

1.  `Reflection`: This class structures the self-critique, requiring the agent to identify what information is `missing` and what is `superfluous` (unnecessary).
2.  `AnswerQuestion`: This class structures the entire response. It forces the agent to provide its main `answer`, a `reflection` (using the `Reflection` class), and a list of `search_queries`.


In [ ]:
class Reflection(BaseModel):
	missing: str = Field(description="What information is missing")
	superfluous: str = Field(description="What information is unnecessary")

class AnswerQuestion(BaseModel):
	answer: str = Field(description="Main response to the question")
	reflection: Reflection = Field(description="Self-critique of the answer")
	search_queries: List[str] = Field(description="Queries for additional research")

#### Binding Tools to the Responder

Now, we bind the `AnswerQuestion` data model as a **tool** to our LLM chain. This crucial step forces the LLM to generate its output in the exact JSON format defined by our Pydantic classes. The LLM doesn't just write text; it calls this "tool" to structure its entire thought process.

After invoking this new chain, we can see the structured output, including the initial answer, the self-critique, and the generated search queries:


In [ ]:
initial_chain = first_responder_prompt| llm.bind_tools(tools=[AnswerQuestion])
response=initial_chain.invoke({"messages":[HumanMessage(question)]})
print("---Full Structured Output---")
print(response.tool_calls)

In [ ]:
answer_content = response.tool_calls[0]['args']['answer']
print("---Initial Answer---")
print(answer_content)

In [ ]:
Reflection_content = response.tool_calls[0]['args']['reflection']
print("---Reflection Answer---")
print(Reflection_content)

In [ ]:
search_queries = response.tool_calls[0]['args']['search_queries']
print("---Search Queries---")
print(search_queries)

### Tool Execution

Now that the Responder has generated search queries based on its self-critique, the next step is to actually *execute* those searches. We'll define a function, `execute_tools`, that takes the agent's state, extracts the search queries, runs them through the Serper tool, and returns the results.

We will also manage the conversation history in `response_list`:


In [ ]:
response_list=[]
response_list.append(HumanMessage(content=question))
response_list.append(response)

In [ ]:
tool_call=response.tool_calls[0]
search_queries = tool_call["args"].get("search_queries", [])
print(search_queries)

In [ ]:
def execute_tools(state: List[BaseMessage]) -> List[BaseMessage]:
    """Run every search query the model asked for and return the results as ToolMessages."""
    last_ai_message = state[-1]
    tool_messages = []

    for tool_call in last_ai_message.tool_calls:
        if tool_call["name"] in ["AnswerQuestion", "ReviseAnswer"]:
            call_id = tool_call["id"]
            search_queries = tool_call["args"].get("search_queries", [])

            # Collect results for each query the model proposed.
            query_results = {query: web_search(query) for query in search_queries}

            tool_messages.append(
                ToolMessage(
                    content=json.dumps(query_results),
                    tool_call_id=call_id,
                    name=tool_call["name"],
                )
            )

    return tool_messages

In [ ]:
tool_response = execute_tools(response_list)
# Use .extend() to add all tool messages from the list
response_list.extend(tool_response)

In [ ]:
tool_response

In [ ]:
response_list

### Defining the Revisor

The **Revisor** is the final piece of the Reflection loop. Its job is to take the original answer, the self-critique, and the new information from the tool search, and then generate an improved, more evidence-based response.

We create a new set of instructions (`revise_instructions`) that guide the Revisor. These instructions emphasize:
- Incorporating the critique.
- Adding numerical citations from the research.
- Distinguishing between correlation and causation.
- Adding a "References" section.


In [ ]:
revise_instructions = """Revise your previous answer using the new information you gathered.
- Incorporate the previous critique, focusing on mechanism and individual variability.
- You MUST include numerical citations pointing to the sources you actually retrieved. Do not
  invent references; cite only URLs that appeared in the search results.
- Distinguish correlation from causation, and acknowledge limitations in the current research.
- Mention relevant biomarkers (for example lipid panels or inflammatory markers) where useful.
- Add a "References" section at the bottom (it does not count toward the word limit), formatted:
- [1] https://example.com
- [2] https://example.com
- Use the critique to remove speculation and keep only claims the evidence supports.
- Keep the response under 250 words: precision over volume.
"""
revisor_prompt = prompt_template.partial(first_instruction=revise_instructions)

#### Structuring the Revisor's Output

Just as we did with the Responder, we define a Pydantic class, `ReviseAnswer`, to structure the Revisor's output. This class inherits from `AnswerQuestion` but adds a new field for `references`, ensuring the agent includes citations in its revised answer.

We then bind this new tool to the revisor chain:


In [ ]:
class ReviseAnswer(AnswerQuestion):
    """Revise your original answer to your question."""
    references: List[str] = Field(description="Citations motivating your updated answer.")
revisor_chain = revisor_prompt | llm.bind_tools(tools=[ReviseAnswer])

#### Invoking the Revisor

Finally, we invoke the `revisor_chain`, passing it the entire conversation history: the original question, the first response (with its critique and search queries), and the new information gathered from the tool search. This provides the Revisor with all the context it needs to generate a final, improved answer.


In [ ]:
response = revisor_chain.invoke({"messages": response_list})
print("---Revised Answer with References---")
print(response.tool_calls[0]['args'])

In [ ]:
response_list.append(response)

## Building the Graph

Now we will use **LangGraph** to assemble these components—Responder, Tool Executor, and Revisor—into a cohesive, cyclical workflow. A graph is a natural way to represent this process, where nodes represent the different stages of thinking and edges represent the flow of information between them.

### Defining the Event Loop

The core of our graph is the event loop. This function determines whether the agent should continue its revision process or if it has reached a satisfactory conclusion. We'll set a maximum number of iterations to prevent the agent from getting stuck in an infinite loop:


In [ ]:
MAX_ITERATIONS = 4

In [ ]:
def event_loop(state: List[BaseMessage]) -> str:
    count_tool_visits = sum(isinstance(item, ToolMessage) for item in state)
    num_iterations = count_tool_visits
    if num_iterations >= MAX_ITERATIONS:
        return END
    return "execute_tools"

In [ ]:
graph=MessageGraph()

graph.add_node("respond", initial_chain)
graph.add_node("execute_tools", execute_tools)
graph.add_node("revisor", revisor_chain)

In [ ]:
graph.add_edge("respond", "execute_tools")
graph.add_edge("execute_tools", "revisor")

In [ ]:
graph.add_conditional_edges("revisor", event_loop)
graph.set_entry_point("respond")

## Running the Agent

With our graph compiled, we're ready to run the full Reflection agent. We'll give it a new, more complex query that requires careful, evidence-based advice.

As the agent runs, we can see the entire process unfold: the initial draft, the self-critique, the tool searches, and the final, revised answer that incorporates the new evidence.


In [ ]:
app = graph.compile()

responses = app.invoke(
    "What does the evidence say about breakfast choices for managing blood sugar?"
)

In [ ]:
print("--- Initial Draft Answer ---")
initial_answer = responses[1].tool_calls[0]['args']['answer']
print(initial_answer)
print("\n")

print("--- Intermediate and Final Revised Answers ---")
answers = []

# Loop through all messages in reverse to find all tool_calls with answers
for msg in reversed(responses):
    if getattr(msg, 'tool_calls', None):
        for tool_call in msg.tool_calls:
            answer = tool_call.get('args', {}).get('answer')
            if answer:
                answers.append(answer)

# Print all collected answers
for i, ans in enumerate(answers):
    label = "Final Revised Answer" if i == 0 else f"Intermediate Step {len(answers) - i}"
    print(f"{label}:\n{ans}\n")


## Author

**Anas AlGhannam**  
[github.com/AnasAlghannam](https://github.com/AnasAlghannam)